# Course: Data Science Course
# Author: Sandro Camargo <sandrocamargo@unipampa.edu.br>
# Class 03 - Multiple Linear Regression Datasus


To open this code in your Google Colab environment, [click here](https://colab.research.google.com/github/Sandrocamargo/data-science/blob/master/cd04_RegressaoLinearMultipla_Melanoma.ipynb).

# Carga de bibliotecas

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.metrics import r2_score
import statsmodels.api as sm
import math
import geopandas as gpd
import seaborn as sns

# Carga de dados obtidos a partir do portal DATASUS

* Morbidade Hospitalar do SUS - por local de residência - Rio Grande do Sul
* Linha: Por região e UF
* Coluna: Por ano de atendimento
* Conteúdo: Internações
* Internações por Ano atendimento segundo Região de Saúde (CIR)
* Capítulo CID-10: II. Neoplasias (tumores)
* Lista Morb CID-10: Neoplasia maligna da pele
* Período: Jan/2008-Dez/2025
* colunas separadas por ;


https://datasus.saude.gov.br/acesso-a-informacao/morbidade-hospitalar-do-sus-sih-sus/


In [ ]:
df = pd.read_csv("https://raw.githubusercontent.com/Sandrocamargo/data-science/refs/heads/main/datasets/melanoma-br-2008-2025.txt", sep=";", na_values="-")
df.info()

dados_estados = pd.read_csv("https://raw.githubusercontent.com/Sandrocamargo/data-science/refs/heads/main/datasets/dados-estados.csv")
dados_estados.info()
#print(dados_estados[dados_estados['Regiao'] == "Rio Grande do Sul"])

In [ ]:
novo_df = df[['Região/Unidade da Federação', '2023', '2024','2025']].copy()
novo_df['Região/Unidade da Federação'] = novo_df['Região/Unidade da Federação'].str.replace('.. ', '', regex=False)
print(novo_df)

dados_estados['Latitude_Capital'] = dados_estados['Latitude_Capital'].abs()

In [ ]:
df_final = pd.merge(novo_df, dados_estados, left_on='Região/Unidade da Federação', right_on='Regiao', how='inner')
df_final['Casos_por_milhao']=(1000000*(df_final['2025']+df_final['2024']+df_final['2023'])/3)/df_final['Pop_2022']
df_final.drop(['2023', '2024','2025'], axis=1, inplace=True)
#df_final['Populacao_Branca'] = df_final['Populacao_Branca'] / 100 * df_final['Pop_2022']
#df_final['Populacao_Parda'] = df_final['Populacao_Parda'] / 100 * df_final['Pop_2022']
#f_final['Populacao_Preta'] = df_final['Populacao_Preta'] / 100 * df_final['Pop_2022']
#df_final['Populacao_Indigena'] = df_final['Populacao_Indigena'] / 100 * df_final['Pop_2022']
#df_final['Populacao_Amarela'] = df_final['Populacao_Amarela'] / 100 * df_final['Pop_2022']
df_final.info()
df_final[df_final['Regiao']=="Rio Grande do Sul"]

In [ ]:
print(df_final[['Sigla','Casos_por_milhao']].sort_values(by='Sigla', ascending=True))

In [ ]:
# matriz de correlação
corr = df_final.select_dtypes(include=['float64', 'int64']).corr()

# ordenar variáveis pela correlação com a variável-alvo
ordem = corr['Casos_por_milhao'].sort_values(ascending=False).index

# reorganizar linhas e colunas
corr_ordenada = corr.loc[ordem, ordem]

#print(corr)

plt.figure(figsize=(12,8))

sns.heatmap(
    corr_ordenada,
    annot=True,
    cmap='coolwarm',
    fmt='.2f'
)

plt.title('Matriz de Correlação')
plt.savefig('matriz_correlacao.pdf', bbox_inches='tight')
plt.show()

In [ ]:
sns.clustermap(corr, cmap='coolwarm', annot=True, figsize=(12,12))
plt.savefig('cluster_map.pdf', bbox_inches='tight')

In [ ]:
# ==========================================
# O dataframe: df_final
# Deve conter:
# - coluna 'Sigla'
# - coluna 'Casos_por_milhao'
# ==========================================

# ------------------------------------------
# 1. Baixar shapefile dos estados do Brasil
# ------------------------------------------

url = "https://raw.githubusercontent.com/codeforamerica/click_that_hood/master/public/data/brazil-states.geojson"

brasil = gpd.read_file(url)

# ------------------------------------------
# 2. Conferir colunas do shapefile
# ------------------------------------------

print(brasil.columns)

# O shapefile possui a sigla dos estados em:
# 'sigla'

# ------------------------------------------
# 3. Padronizar siglas
# ------------------------------------------

brasil["sigla"] = brasil["sigla"].str.upper()
df_final["Sigla"] = df_final["Sigla"].str.upper()

# ------------------------------------------
# 4. Fazer merge
# ------------------------------------------

mapa = brasil.merge(
    df_final,
    left_on="sigla",
    right_on="Sigla",
    how="left"
)

# ------------------------------------------
# 5. Plotar mapa
# ------------------------------------------

fig, ax = plt.subplots(figsize=(12, 12))

mapa.plot(
    column="Casos_por_milhao",
    cmap="Reds",          # escala de cores
    linewidth=0.8,
    edgecolor="black",
    legend=True,
    ax=ax,
    missing_kwds={
        "color": "lightgrey",
        "label": "Sem dados"
    }
)

# Remover eixos
ax.set_axis_off()

# Título
ax.set_title(
    "Morbidade por melanoma por milhão de habitantes nos estados do Brasil",
    fontsize=16,
    pad=20
)

# Opcional: adicionar siglas no mapa
for idx, row in mapa.iterrows():
    if row.geometry.centroid.is_empty:
        continue

    plt.annotate(
        text=row["sigla"],
        xy=(row.geometry.centroid.x, row.geometry.centroid.y),
        horizontalalignment='center',
        fontsize=8
    )

plt.tight_layout()
plt.savefig('mapa.pdf', bbox_inches='tight')
plt.show()

In [ ]:
# Removido por colinearidade 'Populacao_Parda','Populacao_Preta','Populacao_Indigena','Populacao_Amarela'
# Suspeita de colinearidade 2025
X = df_final.drop(['Região/Unidade da Federação','Regiao','Sigla','Casos_por_milhao','Populacao_Parda','Populacao_Preta','Populacao_Indigena','Populacao_Amarela','Pop_2022'], axis=1)
y_target = df_final['Casos_por_milhao']

modelo = LinearRegression()

modelo.fit(X, y_target)

# Predições
predicoes = modelo.predict(X)

In [ ]:
for name, coef in zip(X.columns, modelo.coef_):
    print(f"{name}: {coef}")

# Criando um dataframe com variáveis, coeficientes e média das variáveis
df_coeficientes = pd.DataFrame({
    "Variavel": X.columns,
    "Coeficiente": modelo.coef_,
    "Media_X": X.mean().values
})

df_coeficientes['Produto'] = df_coeficientes['Coeficiente'] * df_coeficientes['Media_X']

# Exibindo o dataframe
print(df_coeficientes)

In [ ]:
r2 = r2_score(y_target, predicoes)

print("\nQUALIDADE DO MODELO")
print(f"R² = {r2:.6f}")

In [ ]:
plt.figure(figsize=(8, 8))

# Dispersão
plt.scatter(
    y_target,
    predicoes,
    alpha=0.5
)

# Linha ideal
valor_min = min(y_target.min(), predicoes.min())
valor_max = max(y_target.max(), predicoes.max())

plt.plot(
    [valor_min, valor_max],
    [valor_min, valor_max],
    linewidth=2,
    color='red'
)

# Configurações do gráfico
plt.xlabel("Valores Reais")
plt.ylabel("Valores Estimados")
plt.title("Regressão Linear Múltipla (Modelo completo)\nValores Reais vs Estimados")

# Mostrar R² no gráfico
plt.text(
    valor_min,
    valor_max * 0.9,
    f"R² = {r2:.4f}",
    fontsize=12
)

plt.grid(True)
plt.savefig('modelocompleto.pdf', bbox_inches='tight')
plt.show()

In [ ]:
import statsmodels.api as sm

# Adicionar constante (intercepto)
X_const = sm.add_constant(X)

# Ajustar modelo
modelo = sm.OLS(y_target, X_const).fit()

# Predições + intervalos
pred = modelo.get_prediction(X_const)
pred_summary = pred.summary_frame(alpha=0.05)

# Extrair valores
predicoes = pred_summary['mean']
lower = pred_summary['obs_ci_lower']
upper = pred_summary['obs_ci_upper']

In [ ]:
# Ordenar pelo eixo X (y_target)
idx = np.argsort(y_target)

y_sorted = y_target.iloc[idx]
pred_sorted = predicoes.iloc[idx]
lower_sorted = lower.iloc[idx]
upper_sorted = upper.iloc[idx]

In [ ]:
plt.figure(figsize=(8, 8))

# Scatter
plt.scatter(y_target, predicoes, alpha=0.5)

# Linha ideal
valor_min = min(y_target.min(), predicoes.min())
valor_max = max(y_target.max(), predicoes.max())

plt.plot(
    [valor_min, valor_max],
    [valor_min, valor_max],
    linewidth=2
)

# Intervalo correto (ordenado)
plt.fill_between(
    y_sorted,
    lower_sorted,
    upper_sorted,
    alpha=0.2,
    label='95% Prediction Interval'
)

# R²
plt.text(
    valor_min,
    valor_max * 0.9,
    f"R² = {modelo.rsquared:.4f}",
    fontsize=12
)

plt.xlabel("Valores Reais")
plt.ylabel("Valores Estimados")
plt.title("Regressão Linear Múltipla\nValores Reais vs Estimados")

plt.legend()
plt.grid(True)

plt.savefig('modelocompleto.pdf', bbox_inches='tight')
plt.show()

In [ ]:
plt.errorbar(
    y_target,
    predicoes,
    yerr=(upper - predicoes),
    fmt='o',
    alpha=0.5
)

In [ ]:
# Adiciona coluna de intercepto
X_sm = sm.add_constant(X)

modelo_sm = sm.OLS(y_target, X_sm).fit()

print("\n================ MODEL SUMMARY ================\n")

print(modelo_sm.summary())


In [ ]:
modelo = LinearRegression()

sfs = SequentialFeatureSelector(modelo, direction='forward')

sfs.fit(X, y_target)
X_selected = sfs.transform(X)

# Check which features were kept
print("Selected features:", sfs.get_support())

sfs.transform(X).shape

# Lista as pontuações médias obtidas em cada etapa da seleção
print("Pontuações de cada etapa:", sfs.n_features_to_select_)

# 2. Filtre a matriz X mantendo apenas as colunas onde o valor é True
if hasattr(X, "iloc"):
    X2 = X.loc[:, sfs.get_support()]  # Se X for DataFrame Pandas
else:
    X2 = X[:, sfs.get_support()]       # Se X for Array NumPy

# 3. Instancie e treine o seu novo modelo de regressão
novo_modelo = LinearRegression()
novo_modelo.fit(X2, y_target)

print("Novo modelo treinado com sucesso!")
print(f"Formato da nova matriz de treino: {X2.shape}")

# Adiciona coluna de intercepto
X2_sm = sm.add_constant(X2)

modelo_sm = sm.OLS(y_target, X2_sm).fit()

print("\n================ MODEL SUMMARY ================\n")

print(modelo_sm.summary())

In [ ]:
X3 = df_final[['Populacao_Branca','Cob_At_Bas']]
y_target = df_final['Casos_por_milhao']

modelo = LinearRegression()

modelo.fit(X3, y_target)

# Predições
predicoes = modelo.predict(X3)

In [ ]:
r2 = r2_score(y_target, predicoes)

print("\nQUALIDADE DO MODELO")
print(f"R² = {r2:.6f}")

In [ ]:
# 3. Instancie e treine o seu novo modelo de regressão
novo_modelo2 = LinearRegression()
novo_modelo2.fit(X3, y_target)

print("Novo modelo treinado com sucesso!")
print(f"Formato da nova matriz de treino: {X3.shape}")

# Adiciona coluna de intercepto
X3_sm = sm.add_constant(X3)

modelo_sm = sm.OLS(y_target, X3_sm).fit()

print("\n================ MODEL SUMMARY ================\n")

print(modelo_sm.summary())

In [ ]:
plt.figure(figsize=(8, 8))

# Dispersão
plt.scatter(
    y_target,
    predicoes,
    alpha=0.5
)

# Linha ideal
valor_min = min(y_target.min(), predicoes.min())
valor_max = max(y_target.max(), predicoes.max())

plt.plot(
    [valor_min, valor_max],
    [valor_min, valor_max],
    linewidth=2,
    color='red'
)

# Configurações do gráfico
plt.xlabel("Valores Reais")
plt.ylabel("Valores Estimados")
plt.title("Regressão Linear Múltipla (Modelo reduzido)\nValores Reais vs Estimados")

# Mostrar R² no gráfico
plt.text(
    valor_min,
    valor_max * 0.9,
    f"R² = {r2:.4f}",
    fontsize=12
)

plt.grid(True)
plt.savefig('modelominimo.pdf', bbox_inches='tight')
plt.show()

In [ ]:
cols_remover = ['Região/Unidade da Federação', 'Regiao', 'Sigla']
df = df_final.drop(columns=cols_remover)

In [ ]:
X = df.drop(columns=['Casos_por_milhao']).values
y = df['Casos_por_milhao'].values

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [ ]:
from sklearn.cross_decomposition import PLSRegression

pls = PLSRegression(n_components=3)  # pode testar 2–4
pls.fit(X_scaled, y)

In [ ]:
def calcular_vip(pls, X):
    t = pls.x_scores_
    w = pls.x_weights_
    q = pls.y_loadings_

    p, h = w.shape
    vip = np.zeros(p)

    s = np.diag(t.T @ t @ q.T @ q).reshape(h, -1)
    total_s = np.sum(s)

    for i in range(p):
        weight = np.array([
            (w[i, j] ** 2) * s[j] for j in range(h)
        ])
        vip[i] = np.sqrt(p * np.sum(weight) / total_s)

    return vip

In [ ]:
vip_scores = calcular_vip(pls, X_scaled)

import pandas as pd

resultado_vip = pd.DataFrame({
    'Variavel': df.drop(columns=['Casos_por_milhao']).columns,
    'VIP': vip_scores
}).sort_values(by='VIP', ascending=False)

print(resultado_vip)

In [ ]:
plt.figure()
plt.barh(resultado_vip['Variavel'], resultado_vip['VIP'])
plt.axvline(x=1.0)
plt.gca().invert_yaxis()
plt.title('VIP - Importância das Variáveis')
plt.savefig('vip_importance.pdf')
plt.show()